# Task 2A: Depot Demand Forecasting

This notebook builds a time-series forecasting model for depot demand. We forecast the total volume and chilled volume for each depot and brand over 10 future weeks (2026 weeks 14-23).

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import load_deliveries_train, load_task1_test_inputs, load_calendar, load_submission_template
from src.features_task2a import create_task2a_features

PROJECT_ROOT = Path('..').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

## 1. Data Loading and Preparation

In [2]:
print('Loading data...')
train_del = load_deliveries_train()
test1_del = load_task1_test_inputs()
calendar = load_calendar()

# Get historical aggregated demand (2024 W1 to 2026 W13)
history_df = create_task2a_features(train_del, test1_del, calendar)
print(f'Historical data shape: {history_df.shape}')
print(history_df.head())

Loading data...


Historical data shape: (702, 6)
   depot  brand  iso_year  iso_week  total_volume_m3  chilled_volume_m3
0  Kandy  Fresh      2024         1          462.279            165.348
1  Kandy  Fresh      2024         2          507.876            180.584
2  Kandy  Fresh      2024         3          460.351            162.967
3  Kandy  Fresh      2024         4          494.885            179.141
4  Kandy  Fresh      2024         5          457.753            166.444


We need to build a complete grid of Depot x Brand x Week from 2024 W1 to 2026 W23.

In [3]:
# Create the calendar weekly features
cal_weekly = calendar.groupby(['iso_year', 'iso_week']).agg(
    days_in_week=('date', 'count'),
    paydays=('is_payday', 'sum'),
    holidays=('is_holiday', 'sum'),
    monsoon_days=('monsoon', 'sum'),
    operating_days=('is_operating', 'sum'),
    avg_festival_ramp=('festival_ramp', 'mean')
).reset_index()

# Generate the complete grid
depots = ['Peliyagoda', 'Kandy']
brands = ['Fresh', 'Style', 'Tech']
years_weeks = cal_weekly[['iso_year', 'iso_week']].drop_duplicates().values

grid = []
for d in depots:
    for b in brands:
        for y, w in years_weeks:
            grid.append({'depot': d, 'brand': b, 'iso_year': y, 'iso_week': w})

grid_df = pd.DataFrame(grid)

# Filter out weeks past 2026 W23 since we don't need to predict beyond that
grid_df = grid_df[(grid_df['iso_year'] < 2026) | ((grid_df['iso_year'] == 2026) & (grid_df['iso_week'] <= 23))]

# Merge with historical actuals
df = grid_df.merge(history_df, on=['depot', 'brand', 'iso_year', 'iso_week'], how='left')

# Merge with calendar features
df = df.merge(cal_weekly, on=['iso_year', 'iso_week'], how='left')

# Encode categoricals
df['depot_enc'] = df['depot'].astype('category').cat.codes
df['brand_enc'] = df['brand'].astype('category').cat.codes

print(f'Full grid shape: {df.shape}')
display(df[df['total_volume_m3'].isna()].head(3))

Full grid shape: (762, 14)


,depot,brand,iso_year,iso_week,total_volume_m3,chilled_volume_m3,days_in_week,paydays,holidays,monsoon_days,operating_days,avg_festival_ramp,depot_enc,brand_enc
117,Peliyagoda,Fresh,2026,14,NaN,NaN,7,1,0,7,6,0.042857,1,0
118,Peliyagoda,Fresh,2026,15,NaN,NaN,7,0,0,7,6,0.600000,1,0
119,Peliyagoda,Fresh,2026,16,NaN,NaN,7,0,2,7,4,0.142857,1,0


## 2. Recursive Forecasting Model
We will train a LightGBM model on data up to 2026 W13. Then we will recursively predict W14 to W23, updating the lag features at each step.

In [4]:
def compute_lags(data: pd.DataFrame):
    """Compute lag features based on the current state of total_volume_m3 and chilled_volume_m3"""
    df = data.sort_values(['depot', 'brand', 'iso_year', 'iso_week']).copy()
    
    for i in [1, 2, 3, 4, 8, 52]:
        df[f'lag_{i}_total'] = df.groupby(['depot', 'brand'])['total_volume_m3'].shift(i)
        df[f'lag_{i}_chilled'] = df.groupby(['depot', 'brand'])['chilled_volume_m3'].shift(i)
        
    # Rolling means
    df['roll_4_total'] = df.groupby(['depot', 'brand'])['lag_1_total'].rolling(4, min_periods=1).mean().reset_index(level=[0,1], drop=True)
    df['roll_4_chilled'] = df.groupby(['depot', 'brand'])['lag_1_chilled'].rolling(4, min_periods=1).mean().reset_index(level=[0,1], drop=True)
    
    return df

features = [
    'depot_enc', 'brand_enc', 'iso_week', 'paydays', 'holidays', 
    'monsoon_days', 'operating_days', 'avg_festival_ramp',
    'lag_1_total', 'lag_2_total', 'lag_3_total', 'lag_4_total', 'lag_8_total', 'lag_52_total', 'roll_4_total'
]

chilled_features = [
    'depot_enc', 'iso_week', 'paydays', 'holidays', 
    'monsoon_days', 'operating_days', 'avg_festival_ramp',
    'lag_1_chilled', 'lag_2_chilled', 'lag_3_chilled', 'lag_4_chilled', 'lag_8_chilled', 'lag_52_chilled', 'roll_4_chilled'
]

In [5]:
# Initial lag computation on history
df_features = compute_lags(df)

# Train-validation split for initial evaluation (optional, here we train on all history)
train_mask = (df_features['iso_year'] < 2026) | ((df_features['iso_year'] == 2026) & (df_features['iso_week'] <= 13))

train_df = df_features[train_mask].dropna(subset=['lag_4_total'])

# Model for Total Volume
model_total = lgb.LGBMRegressor(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=5, 
    random_state=42
)
model_total.fit(train_df[features], train_df['total_volume_m3'])

# Model for Chilled Volume (Only trained on Fresh brand)
train_df_fresh = train_df[train_df['brand'] == 'Fresh']
model_chilled = lgb.LGBMRegressor(
    n_estimators=150, 
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
model_chilled.fit(train_df_fresh[chilled_features], train_df_fresh['chilled_volume_m3'])

[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000115 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1576
[LightGBM] [Info] Number of data points in the train set: 678, number of used features: 15
[LightGBM] [Info] Start training from score 288.604068
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No

LGBMRegressor(learning_rate=0.05, max_depth=4, n_estimators=150,
              random_state=42)

## 3. Generate Forecast (Weeks 14-23)

In [6]:
forecast_df = df.copy()

# Forecast step-by-step
for week in range(14, 24):
    # Recompute lags with current state of forecast_df
    current_state = compute_lags(forecast_df)
    
    # Filter to target week
    target_mask = (current_state['iso_year'] == 2026) & (current_state['iso_week'] == week)
    X_target = current_state[target_mask]
    
    if len(X_target) == 0:
        continue
        
    # Predict total volume
    pred_total = model_total.predict(X_target[features])
    
    # Predict chilled volume (only for Fresh)
    pred_chilled = np.zeros(len(X_target))
    is_fresh = X_target['brand'] == 'Fresh'
    if is_fresh.any():
        pred_chilled[is_fresh] = model_chilled.predict(X_target.loc[is_fresh, chilled_features])
        
    # Update forecast_df
    forecast_df.loc[target_mask, 'total_volume_m3'] = np.maximum(0, pred_total)
    forecast_df.loc[target_mask, 'chilled_volume_m3'] = np.maximum(0, pred_chilled)

print('Forecasting complete.')

[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^m

## 4. Prepare Submission

In [7]:
test2a = pd.read_csv(PROJECT_ROOT / 'data/raw/Test Data/task2a_test_inputs.csv')

submission = test2a.merge(
    forecast_df[['depot', 'brand', 'iso_year', 'iso_week', 'total_volume_m3', 'chilled_volume_m3']],
    on=['depot', 'brand', 'iso_year', 'iso_week'],
    how='left'
)

# Format according to rules
submission.rename(columns={
    'total_volume_m3': 'pred_total_volume_m3',
    'chilled_volume_m3': 'pred_chilled_volume_m3'
}, inplace=True)

# Ensure Style and Tech have strictly 0 chilled volume
submission.loc[submission['brand'] != 'Fresh', 'pred_chilled_volume_m3'] = 0.0

out_cols = ['row_id', 'pred_total_volume_m3', 'pred_chilled_volume_m3']
submission_final = submission[out_cols]

print(submission_final.head(10))

out_path = PROJECT_ROOT / 'data/raw/Submission Templates/submission_task2a.csv'
submission_final.to_csv(out_path, index=False)
print(f'\nSaved to {out_path}')

# Save the models
model_total.booster_.save_model(MODELS_DIR / 'task2a_total_volume_lgb.txt')
model_chilled.booster_.save_model(MODELS_DIR / 'task2a_chilled_volume_lgb.txt')

  row_id  pred_total_volume_m3  pred_chilled_volume_m3
0  W0000           1037.356580              381.870211
1  W0001            135.407202                0.000000
2  W0002             44.104220                0.000000
3  W0003            548.769017              193.859202
4  W0004             88.448567                0.000000
5  W0005             31.568479                0.000000
6  W0006           1093.028601              410.208600
7  W0007            189.795407                0.000000
8  W0008             60.156115                0.000000
9  W0009            756.444932              234.224846

Saved to C:\Users\sithu\MyWorks\My Softwares\Competitions\Tech_Triathlon_26\TT_Datathon\data\raw\Submission Templates\submission_task2a.csv
